In [1]:
import numpy as np
import polars as pl
import altair as alt
import duckdb
import ipywidgets as widgets
from IPython.display import display

In [2]:
n = 10

In [3]:
squares = [i**2 for i in range(n)]
squares

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]

In [4]:
def plot_sine(freq=2.0, amp=1.0, noise=False):
    x = np.linspace(0, 2 * np.pi, 400)
    y = amp * np.sin(freq * x)
    if noise:
        y = y + np.random.default_rng(0).normal(0, 0.15, x.size)
    display(
        alt.Chart(pl.DataFrame({"x": x, "y": y})).mark_line().encode(
            x="x", y=alt.Y("y", scale=alt.Scale(domain=[-3.5, 3.5]))
        ).properties(width=600, height=250)
    )


widgets.interact(
    plot_sine,
    freq=widgets.FloatSlider(value=2, min=0.5, max=10, step=0.5, description="frequency"),
    amp=widgets.FloatSlider(value=1, min=0.1, max=3, step=0.1, description="amplitude"),
    noise=widgets.Checkbox(value=False, description="add noise"),
);

interactive(children=(FloatSlider(value=2.0, description='frequency', max=10.0, min=0.5, step=0.5), FloatSlide…

In [5]:
from sklearn.datasets import load_iris

_iris = load_iris(as_frame=True)
df = pl.from_pandas(_iris.frame).rename(
    lambda c: c.replace(" (cm)", "").replace(" ", "_")
).with_columns(
    species=pl.col("target").replace_strict(
        dict(enumerate(_iris.target_names)), return_dtype=pl.String
    )
).drop("target")
df

sepal_length,sepal_width,petal_length,petal_width,species
f64,f64,f64,f64,str
5.1,3.5,1.4,0.2,"""setosa"""
4.9,3.0,1.4,0.2,"""setosa"""
4.7,3.2,1.3,0.2,"""setosa"""
4.6,3.1,1.5,0.2,"""setosa"""
5.0,3.6,1.4,0.2,"""setosa"""
…,…,…,…,…
6.7,3.0,5.2,2.3,"""virginica"""
6.3,2.5,5.0,1.9,"""virginica"""
6.5,3.0,5.2,2.0,"""virginica"""


In [6]:
brush = alt.selection_interval()
alt.Chart(df).mark_circle(size=60).encode(
    x="petal_length",
    y="petal_width",
    color=alt.condition(brush, "species", alt.value("lightgray")),
).add_params(brush).properties(height=300)

alt.Chart(...)

In [7]:
# Altair selections stay in the browser: Python can't read them back,
# so filter by hand to get the selected rows
selected = df.filter(
    pl.col("petal_length").is_between(1.0, 5.0)
    & pl.col("petal_width").is_between(0.1, 1.5)
)
print(f"{len(selected)} flowers selected, species: "
      f"{dict(selected['species'].value_counts().iter_rows())}")
selected

96 flowers selected, species: {'virginica': 1, 'versicolor': 45, 'setosa': 50}


sepal_length,sepal_width,petal_length,petal_width,species
f64,f64,f64,f64,str
5.1,3.5,1.4,0.2,"""setosa"""
4.9,3.0,1.4,0.2,"""setosa"""
4.7,3.2,1.3,0.2,"""setosa"""
4.6,3.1,1.5,0.2,"""setosa"""
5.0,3.6,1.4,0.2,"""setosa"""
…,…,…,…,…
5.7,2.9,4.2,1.3,"""versicolor"""
6.2,2.9,4.3,1.3,"""versicolor"""
5.1,2.5,3.0,1.1,"""versicolor"""


In [8]:
min_petal = widgets.FloatSlider(value=4, min=1, max=7, step=0.1, description="min petal")
out = widgets.Output()


def run_query(change=None):
    with out:
        out.clear_output()
        display(duckdb.sql(f"""
            SELECT species, COUNT(*) AS flowers
            FROM df
            WHERE petal_length >= {min_petal.value}
            GROUP BY species
        """).pl())


min_petal.observe(run_query, names="value")
run_query()
display(min_petal, out)

FloatSlider(value=4.0, description='min petal', max=7.0, min=1.0)

Output()

In [9]:
# No built-in dataframe editor: write the transform yourself
df.filter(pl.col("petal_length") > 4).group_by("species").agg(pl.all().mean())

species,sepal_length,sepal_width,petal_length,petal_width
str,f64,f64,f64,f64
"""versicolor""",6.147059,2.902941,4.517647,1.411765
"""virginica""",6.588,2.974,5.552,2.026


In [10]:
df.describe()

statistic,sepal_length,sepal_width,petal_length,petal_width,species
str,f64,f64,f64,f64,str
"""count""",150.0,150.0,150.0,150.0,"""150"""
"""null_count""",0.0,0.0,0.0,0.0,"""0"""
"""mean""",5.843333,3.057333,3.758,1.199333,null
"""std""",0.828066,0.435866,1.765298,0.762238,null
"""min""",4.3,2.0,1.0,0.1,"""setosa"""
"""25%""",5.1,2.8,1.6,0.3,null
"""50%""",5.8,3.0,4.4,1.3,null
"""75%""",6.4,3.3,5.1,1.8,null
"""max""",7.9,4.4,6.9,2.5,"""virginica"""


In [11]:
report = (
    df.filter(pl.col("petal_length") >= 4)
    .group_by("species")
    .agg(pl.len().alias("n"), pl.col("sepal_length").mean().round(2).alias("mean_sepal_length"))
    .sort("species")
)
report

species,n,mean_sepal_length
str,u32,f64
"""versicolor""",39,6.1
"""virginica""",50,6.59
